In [ ]:
import cv2
import numpy as np
import sqlite3
import joblib
import mediapipe as mp
from deepface.DeepFace import represent
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Load VGG16 as a backup feature extractor
vgg_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
feature_extractor = Model(inputs=vgg_model.input, outputs=vgg_model.get_layer("block5_pool").output)

# Define fixed feature size
FEATURE_SIZE = 4096  # Change based on feature extractor output

# 🔹 Extract user features (FIXED)
def extract_features(image):
    try:
        features_dict = represent(image, model_name="VGG-Face", enforce_detection=False)
        features = np.array(features_dict[0]["embedding"], dtype=np.float32)
    except:
        img_resized = cv2.resize(image, (224, 224))
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        img_preprocessed = np.expand_dims(img_rgb, axis=0) / 255.0
        features = feature_extractor.predict(img_preprocessed).flatten()  # TensorFlow VGG16 fallback
    
    # 🔹 Ensure fixed size
    features = np.array(features, dtype=np.float32)
    
    if features.shape[0] > FEATURE_SIZE:
        features = features[:FEATURE_SIZE]  # Trim if too large
    elif features.shape[0] < FEATURE_SIZE:
        features = np.pad(features, (0, FEATURE_SIZE - features.shape[0]))  # Pad if too small

    return features

# 🔹 Create database table
def create_table():
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute('DROP TABLE IF EXISTS users')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            name TEXT,
            features BLOB
        )
    ''')
    conn.commit()
    conn.close()

# 🔹 Load user data from the database (FIXED)
def load_user_data():
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users")
    data = cursor.fetchall()
    features, labels = [], []
    
    for row in data:
        name = row[0]
        user_features = np.frombuffer(row[1], dtype=np.float32)
        
        if user_features.shape[0] == FEATURE_SIZE:  # Only include valid embeddings
            features.append(user_features)
            labels.append(name)
    
    conn.close()

    if len(features) == 0:
        return np.array([]), np.array([])
    
    return np.vstack(features), np.array(labels)

# 🔹 Train SVM model (FIXED)
def train_model():
    features, labels = load_user_data()
    
    if len(features) < 2:
        print("Not enough data to train the model.")
        return None

    # Normalize feature vectors
    scaler = StandardScaler()
    features = scaler.fit_transform(features)

    svm_model = SVC(kernel='linear', probability=True)
    svm_model.fit(features, labels)

    joblib.dump((svm_model, scaler), 'svm_model.pkl')  # Save model & scaler
    print("✅ SVM Model trained successfully!")
    
    return svm_model

# 🔹 Predict user identity (FIXED)
def predict_user(image):
    try:
        svm_model, scaler = joblib.load('svm_model.pkl')
    except:
        print("❌ No trained model found!")
        return None
    
    user_features = extract_features(image)
    user_features = scaler.transform([user_features])  # Apply normalization
    predicted_label = svm_model.predict(user_features)
    
    return predicted_label[0]

# 🔹 Face detection using Mediapipe
mp_face_detection = mp.solutions.face_detection

def preprocess_user(image):
    with mp_face_detection.FaceDetection(min_detection_confidence=0.5) as face_detection:
        rgb_frame = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb_frame)

        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                h, w, _ = image.shape
                x, y, w, h = int(bboxC.xmin * w), int(bboxC.ymin * h), int(bboxC.width * w), int(bboxC.height * h)
                face = image[y:y+h, x:x+w]
                face_resized = cv2.resize(face, (224, 224))
                return face_resized

    return None  # No face detected

# 🔹 Capture users from video & store them
def capture_and_store_users():
    video_paths = ["data/criminal/user1.mp4","data/criminal/user2.mp4","data/missing/user3.mp4","data/missing/user4.mp4"]

    for i, video_path in enumerate(video_paths):
        cap = cv2.VideoCapture(video_path)
        user_name = f'user_{i+1}'

        print(f"Processing {user_name}'s video...")
        users = []

        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or frame_count >= 500:
                break
            
            user = preprocess_user(frame)
            if user is not None:
                users.append(user)
                frame_count += 1

        # Store extracted users in the database
        conn = sqlite3.connect('users.db')
        cursor = conn.cursor()

        for user in users:
            features = extract_features(user)
            cursor.execute("INSERT INTO users (name, features) VALUES (?, ?)", 
                           (user_name, features.tobytes()))

        conn.commit()
        conn.close()
        cap.release()
    
    train_model()
    cv2.destroyAllWindows()

# 🔹 Predict user from video & count faces detected
def predict_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    detected_faces = 0
    predictions = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        user_face = preprocess_user(frame)
        if user_face is not None:
            detected_faces += 1
            user_label = predict_user(user_face)
            if user_label:
                predictions.append(user_label)

    cap.release()
    cv2.destroyAllWindows()

    if predictions:
        most_common_user = max(set(predictions), key=predictions.count)
        print(f"Predicted User: {most_common_user}, Detected Faces: {detected_faces}")
        return most_common_user, detected_faces
    else:
        print("No face detected in the video.")
        return None, detected_faces

# 🔹 Testing function
def test_prediction(image_path):
    test_image = cv2.imread(image_path)
    processed_test = preprocess_user(test_image)

    if processed_test is not None:
        result = predict_user(processed_test)
        print(f"Predicted User: {result}")
    else:
        print("No face detected!")

# 🔹 Main execution
if __name__ == "__main__":
    create_table()  # Create database table
    capture_and_store_users()  # Capture users from video

    


: 

In [4]:

import cv2
import numpy as np
import sqlite3
import joblib
import mediapipe as mp
from deepface.DeepFace import represent
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
FEATURE_SIZE = 4096  # Change based on feature extractor output
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

mp_face_detection = mp.solutions.face_detection
def predict_user(image):
    try:
        svm_model, scaler = joblib.load('svm_model.pkl')
    except:
        print("❌ No trained model found!")
        return None
    
    user_features = extract_features(image)
    user_features = scaler.transform([user_features])  # Apply normalization
    predicted_label = svm_model.predict(user_features)
    
    return predicted_label[0]


# 🔹 Extract user features (FIXED)
def extract_features(image):
    try:
        features_dict = represent(image, model_name="VGG-Face", enforce_detection=False)
        features = np.array(features_dict[0]["embedding"], dtype=np.float32)
    except:
        img_resized = cv2.resize(image, (224, 224))
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        img_preprocessed = np.expand_dims(img_rgb, axis=0) / 255.0
        features = feature_extractor.predict(img_preprocessed).flatten()  # TensorFlow VGG16 fallback
    
    # 🔹 Ensure fixed size
    features = np.array(features, dtype=np.float32)
    
    if features.shape[0] > FEATURE_SIZE:
        features = features[:FEATURE_SIZE]  # Trim if too large
    elif features.shape[0] < FEATURE_SIZE:
        features = np.pad(features, (0, FEATURE_SIZE - features.shape[0]))  # Pad if too small

    return features
# 🔹 Face detection using Mediapipe
mp_face_detection = mp.solutions.face_detection

def preprocess_user(image):
    with mp_face_detection.FaceDetection(min_detection_confidence=0.5) as face_detection:
        rgb_frame = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb_frame)

        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                h, w, _ = image.shape
                x, y, w, h = int(bboxC.xmin * w), int(bboxC.ymin * h), int(bboxC.width * w), int(bboxC.height * h)
                face = image[y:y+h, x:x+w]
                face_resized = cv2.resize(face, (224, 224))
                return face_resized

    return None  # No face detected
# 🔹 Predict user from video & count faces detected
def predict_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    detected_faces = 0
    predictions = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        user_face = preprocess_user(frame)
        if user_face is not None:
            detected_faces += 1
            user_label = predict_user(user_face)
            
            if user_label:
                print(user_label)
                predictions.append(user_label)

    cap.release()
    cv2.destroyAllWindows()

    if predictions:
        most_common_user = max(set(predictions), key=predictions.count)
        print(f"Predicted User: {most_common_user}, Detected Faces: {detected_faces} totalseconds{detected_faces/20} seconds")
        return most_common_user, detected_faces
    else:
        print("No face detected in the video.")
        return None, detected_faces
# Example test video prediction
test_video = 'user3.mp4'  # Change this path as needed
predict_from_video(test_video)

user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1
user_1

('user_1', 365)